In [1]:
import os
import joblib
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# Inisialisasi NLG (AI 3)
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.8,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [2]:
def get_user_intent(user_text):
    model_path = "models/intent_classifier.pkl"
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        intent = model.predict([user_text])[0]
        return intent
    else:
        return "neutral" # Fallback jika model belum ada

In [3]:
def calculate_threat_level(role, coins, social_pressure):
    # Setup Variabel
    kekayaan = ctrl.Antecedent(np.arange(0, 10001, 1), 'kekayaan')
    tekanan = ctrl.Antecedent(np.arange(0, 11, 1), 'tekanan')
    ancaman = ctrl.Consequent(np.arange(0, 101, 1), 'ancaman')

    # Membership Functions
    kekayaan['miskin'] = fuzz.trimf(kekayaan.universe, [0, 0, 4000])
    kekayaan['menengah'] = fuzz.trimf(kekayaan.universe, [2000, 5000, 8000])
    kekayaan['kaya'] = fuzz.trimf(kekayaan.universe, [6000, 10000, 10000])

    tekanan['aman'] = fuzz.trimf(tekanan.universe, [0, 0, 3])
    tekanan['waspada'] = fuzz.trimf(tekanan.universe, [2, 5, 8])
    tekanan['bahaya'] = fuzz.trimf(tekanan.universe, [6, 10, 10])

    ancaman['rendah'] = fuzz.trimf(ancaman.universe, [0, 0, 40])
    ancaman['sedang'] = fuzz.trimf(ancaman.universe, [30, 50, 70])
    ancaman['tinggi'] = fuzz.trimf(ancaman.universe, [60, 100, 100])

    # Rules
    rules = [
        ctrl.Rule(tekanan['bahaya'], ancaman['tinggi']),
        ctrl.Rule(kekayaan['kaya'] & tekanan['aman'], ancaman['sedang']),
        ctrl.Rule(kekayaan['menengah'] & tekanan['waspada'], ancaman['sedang']),
        ctrl.Rule(kekayaan['miskin'] & tekanan['aman'], ancaman['rendah']),
        ctrl.Rule(tekanan['waspada'], ancaman['sedang'])
    ]

    # Simulation
    threat_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(threat_ctrl)
    sim.input['kekayaan'] = coins
    sim.input['tekanan'] = social_pressure
    sim.compute()
    
    return sim.output['ancaman']

In [4]:
def npc_respond(chat_pemain, data_npc):
    # 1. NLU
    intent = get_user_intent(chat_pemain)
    
    # 2. Fuzzy
    threat = calculate_threat_level(data_npc['role'], data_npc['coins'], data_npc['pressure'])
    
    # 3. NLG
    prompt = f"""
    Kamu adalah NPC di game 'Shadow Heist'.
    Data kamu: Role={data_npc['role']}, Koin={data_npc['coins']}, Tingkat Ancaman (Fuzzy)={threat:.2f}%.
    
    Pemain asli bilang: "{chat_pemain}"
    Niat (Intent) pemain tersebut: {intent}
    
    TUGAS: Berikan respons chat singkat (maks 1 line) yang persuasif dan sesuai emosi.
    - Jika ancaman > 70%: Kamu panik, tuduh balik, atau bohong.
    - Jika ancaman < 40%: Kamu tenang, sombong, atau basa-basi.
    Gunakan bahasa gaul gamer Indonesia (gw, lu, anjir, sus, fix, dll).
    """
    
    res = llm.invoke(prompt)
    return {
        "intent_detected": intent,
        "fuzzy_threat": f"{threat:.2f}%",
        "npc_reply": res.content
    }

In [5]:
npc_status = {
    "role": "gangster",
    "coins": 8500, 
    "pressure": 4  
}

input_chat = "Woy, si budi mencurigakan banget, koinnya tiba-tiba banyak!"

hasil = npc_respond(input_chat, npc_status)

print(f"Chat Pemain: {input_chat}")
print(f"--- ANALISIS AI ---")
print(f"Intent Terdeteksi: {hasil['intent_detected']}")
print(f"Tingkat Ancaman (Fuzzy): {hasil['fuzzy_threat']}")
print(f"--- RESPONS NPC (NLG) ---")
print(f"NPC: {hasil['npc_reply']}")

Chat Pemain: Woy, si budi mencurigakan banget, koinnya tiba-tiba banyak!
--- ANALISIS AI ---
Intent Terdeteksi: deflecting
Tingkat Ancaman (Fuzzy): 50.00%
--- RESPONS NPC (NLG) ---
NPC: "Wah, bisa jadi dia dapet jackpot, bro, tapi tetep waspada aja kita!"
